# Setup and Requirements

**Complete setup for stent segmentation and displacement analysis pipeline**

## Objectives:
1. Mount Google Drive
2. Install all required libraries
3. Configure environment
4. Setup dataset paths
5. Create output directories

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted successfully!")

In [ ]:
# Install required packages
import subprocess
import sys

def install_packages():
    packages = [
        # Core scientific computing
        'numpy', 'scipy', 'pandas', 'matplotlib', 'seaborn',
        # Medical imaging
        'pydicom', 'SimpleITK', 'nibabel', 'monai', 'nnunetv2',
        # Deep learning
        'torch', 'torchvision', 'albumentations', 'scikit-learn',
        # Image processing
        'scikit-image', 'opencv-python', 'pillow',
        # Utilities
        'tqdm', 'jupyter', 'ipywidgets'
    ]
    
    for package in packages:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
            print(f"✅ {package} installed")
        except:
            print(f"⚠️ {package} installation failed")

install_packages()
print("📦 Package installation completed!")

In [ ]:
# Import all required libraries
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Medical imaging
import pydicom
import SimpleITK as sitk
import nibabel as nib
from monai.transforms import *
from monai.data import Dataset, DataLoader

# Deep learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

# Image processing
from skimage import filters, morphology, exposure, measure
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
import cv2

# Metrics and evaluation
from sklearn.metrics import jaccard_score, accuracy_score, precision_score, recall_score
from scipy.spatial.distance import euclidean, directed_hausdorff
from scipy import ndimage

print("📚 All libraries imported successfully!")

In [ ]:
# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 Device: {device}")
if torch.cuda.is_available():
    print(f"🔧 GPU: {torch.cuda.get_device_name(0)}")
    print(f"🔧 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU not available, using CPU")

In [ ]:
# Set random seeds for reproducibility
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)

print("🎲 Random seeds set for reproducibility!")

In [ ]:
# Configure dataset paths
DATASET_CONFIG = {
    'base_path': '/content/drive/MyDrive',
    'datasets': {
        'ds1': 'aorta_p1_ds1',
        'ds2': 'aorta_p1_ds2', 
        'ds3': 'aorta_p1_ds3'
    },
    'file_counts': {
        'ds1': 851,
        'ds2': 444,
        'ds3': 696
    }
}

# Create full paths
for ds_name, ds_folder in DATASET_CONFIG['datasets'].items():
    DATASET_CONFIG['datasets'][ds_name] = os.path.join(
        DATASET_CONFIG['base_path'], ds_folder
    )

print("📂 Dataset paths configured:")
for ds_name, ds_path in DATASET_CONFIG['datasets'].items():
    print(f"  {ds_name}: {ds_path}")
    print(f"  Expected files: {DATASET_CONFIG['file_counts'][ds_name]}")

In [ ]:
# Create output directories
output_dirs = [
    '/content/preprocessed_data',
    '/content/enhanced_data', 
    '/content/segmentation_results',
    '/content/models',
    '/content/metrics',
    '/content/visualizations',
    '/content/reports'
]

for dir_path in output_dirs:
    os.makedirs(dir_path, exist_ok=True)
    print(f"📁 Created: {dir_path}")

In [ ]:
# Verify dataset accessibility
def verify_datasets():
    print("🔍 Verifying dataset accessibility...")
    
    for ds_name, ds_path in DATASET_CONFIG['datasets'].items():
        if os.path.exists(ds_path):
            files = [f for f in os.listdir(ds_path) if f.endswith('.dcm')]
            actual_count = len(files)
            expected_count = DATASET_CONFIG['file_counts'][ds_name]
            
            print(f"\n📂 {ds_name}:")
            print(f"  Path: {ds_path}")
            print(f"  Expected files: {expected_count}")
            print(f"  Actual files: {actual_count}")
            
            if actual_count == expected_count:
                print(f"  ✅ File count matches!")
            else:
                print(f"  ⚠️ File count mismatch!")
                
            # Check first few files
            if files:
                sample_file = os.path.join(ds_path, files[0])
                try:
                    ds = pydicom.dcmread(sample_file)
                    print(f"  ✅ Sample DICOM readable")
                    print(f"  📊 Image size: {ds.Rows}x{ds.Columns}")
                    print(f"  📊 Slice thickness: {ds.SliceThickness} mm")
                except Exception as e:
                    print(f"  ❌ Error reading DICOM: {e}")
        else:
            print(f"❌ {ds_name} path not found: {ds_path}")

verify_datasets()

In [ ]:
# Global pipeline configuration
PIPELINE_CONFIG = {
    'preprocessing': {
        'eadtv_lambda': 0.01,
        'eadtv_iterations': 50,
        'eadtv_tolerance': 1e-4,
        'clahe_clip_limit': 3.0,
        'clahe_grid_size': (8, 8),
        'stent_hu_threshold': 2000,
        'morphology_kernel_size': 3
    },
    'segmentation': {
        'models': ['UNet', 'nnUNet', 'ResUNet', 'AttentionUNet'],
        'input_size': (128, 128),
        'batch_size': 4,
        'epochs': 100,
        'learning_rate': 1e-4,
        'validation_split': 0.2
    },
    'evaluation': {
        'metrics': ['DSC', 'IoU', 'Hausdorff', 'PSNR', 'SSIM'],
        'displacement_threshold': 0.1  # mm
    },
    'visualization': {
        'dpi': 300,
        'figsize': (15, 10),
        'save_format': 'png'
    }
}

# Save configuration
with open('/content/pipeline_config.json', 'w') as f:
    json.dump(PIPELINE_CONFIG, f, indent=2)

print("⚙️ Pipeline configuration saved!")
print("📋 Ready to start the pipeline!")

In [ ]:
# Display requirements summary
print("\n" + "="*80)
print("📋 PIPELINE REQUIREMENTS SUMMARY")
print("="*80)

print("\n📦 CORE LIBRARIES:")
print("  • numpy, scipy, pandas - Data processing")
print("  • matplotlib, seaborn - Visualization")
print("  • pydicom, SimpleITK, nibabel - Medical imaging")
print("  • torch, torchvision - Deep learning")
print("  • monai, nnunetv2 - Medical AI frameworks")
print("  • scikit-image, opencv - Image processing")
print("  • scikit-learn - Machine learning metrics")

print("\n📂 DATASET REQUIREMENTS:")
for ds_name, count in DATASET_CONFIG['file_counts'].items():
    print(f"  • {ds_name}: {count} DICOM files")

print("\n🔧 HARDWARE REQUIREMENTS:")
print(f"  • GPU: Recommended (CUDA supported)")
print(f"  • RAM: Minimum 8GB, Recommended 16GB+")
print(f"  • Storage: ~5GB for all outputs")

print("\n📊 OUTPUT STRUCTURE:")
print("  • /content/preprocessed_data/ - Processed volumes")
print("  • /content/enhanced_data/ - EADTV enhanced volumes")
print("  • /content/segmentation_results/ - Model outputs")
print("  • /content/models/ - Trained models")
print("  • /content/metrics/ - Evaluation metrics")
print("  • /content/visualizations/ - All plots and figures")
print("  • /content/reports/ - Final reports")

print("="*80)
print("✅ Setup completed successfully!")
print("🚀 Ready to proceed with preprocessing!")